# 02 — Cleaning

Build the reproducible Python ETL pipeline for the US Accidents dataset. Every transformation is logged, and the final cleaned dataset is exported to `data/processed/cleaned_dataset.csv` for downstream EDA and statistical analysis.

**Cleaning is organised around the five problem-statement dimensions:**

1. **Severity** — target variable, must be non-null and typed correctly.
2. **Geography** — state, city, lat/lng preserved for hotspot analysis.
3. **Time** — derive hour, day-of-week, month, season, rush-hour, weekend flags.
4. **Weather** — numeric weather variables imputed with medians (robust to skew).
5. **Infrastructure** — POI booleans (traffic signal, junction, crossing, …) forced to bool.

All step-level logic lives in `scripts/etl_pipeline.py` so it can also be run from the CLI.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.etl_pipeline import (
    load_us_accidents,
    drop_unused_columns,
    parse_datetimes,
    drop_missing_critical,
    impute_weather,
    impute_categorical,
    coerce_booleans,
    add_duration,
    add_time_features,
    add_severity_label,
    handle_outliers,
    save_processed,
    DEFAULT_ANALYSIS_SAMPLE_N,
    DROP_COLUMNS,
    NUMERIC_WEATHER_COLUMNS,
    CATEGORICAL_FILL_COLUMNS,
    BOOLEAN_POI_COLUMNS,
)

RAW_PATH       = PROJECT_ROOT / 'data' / 'raw' / 'US_Accidents_March23.csv'
PROCESSED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'cleaned_dataset.csv'

# Tableau Public works best with a smaller extract. We keep the full raw file
# locally, but clean and analyze a reproducible representative sample.
SAMPLE_N: int | None = DEFAULT_ANALYSIS_SAMPLE_N

## Step 1 — Load a representative raw sample & normalize column names

The full combined CSV remains in `data/raw/` for reproducibility. This notebook cleans a fixed 500,000-row random sample so EDA and Tableau stay fast while preserving national-level patterns.

In [2]:
df = load_us_accidents(RAW_PATH, sample_n=SAMPLE_N)
rows_initial, cols_initial = df.shape
print(f'Loaded: {rows_initial:,} rows × {cols_initial} columns')
df.head(3)

Loaded: 500,000 rows × 46 columns


,id,source,severity,start_time,end_time,start_lat,start_lng,end_lat,end_lng,distance_mi,...,roundabout,station,stop,traffic_calming,traffic_signal,turning_loop,sunrise_sunset,civil_twilight,nautical_twilight,astronomical_twilight
0,A-804827,Source2,3,2021-11-24 19:36:02,2021-11-24 20:35:31,32.869389,-96.670082,NaN,NaN,0.000,...,False,False,False,False,False,False,Night,Night,Night,Night
1,A-3362792,Source2,2,2017-08-08 08:52:45,2017-08-08 09:38:00,40.099434,-75.196213,NaN,NaN,0.000,...,False,False,False,False,True,False,Day,Day,Day,Day
2,A-3533324,Source1,2,2016-10-24 16:09:01,2016-10-24 22:09:01,34.143244,-117.256491,34.14364,-117.259669,0.184,...,False,False,False,False,False,False,Day,Day,Day,Day


## Step 2 — Drop columns not used in the government severity analysis

| Column | Reason dropped |
|---|---|
| `id`, `source` | internal identifiers |
| `description` | free text, not structured analysis |
| `end_lat`, `end_lng` | ~50% null, redundant with start coordinates |
| `number` | building number, ~60% null, not analytical |
| `airport_code`, `weather_timestamp` | audit metadata for the weather join |
| `country` | constant = `US` |
| `turning_loop` | constant = `False` in this dataset |
| `nautical_twilight`, `astronomical_twilight` | redundant with `sunrise_sunset` / `civil_twilight` |

In [3]:
df = drop_unused_columns(df)
print(f'After drop: {df.shape[1]} columns')
print('Columns remaining:', list(df.columns))

After drop: 35 columns
Columns remaining: ['severity', 'start_time', 'end_time', 'start_lat', 'start_lng', 'distance_mi', 'street', 'city', 'county', 'state', 'zipcode', 'timezone', 'temperature_f', 'wind_chill_f', 'humidity', 'pressure_in', 'visibility_mi', 'wind_direction', 'wind_speed_mph', 'precipitation_in', 'weather_condition', 'amenity', 'bump', 'crossing', 'give_way', 'junction', 'no_exit', 'railway', 'roundabout', 'station', 'stop', 'traffic_calming', 'traffic_signal', 'sunrise_sunset', 'civil_twilight']


## Step 3 — Parse datetimes and drop rows missing critical fields

`severity`, `start_time`, `state`, `start_lat`, `start_lng` are required for every downstream KPI. Rows missing any of these are dropped.

In [4]:
df = parse_datetimes(df)
before = len(df)
df = drop_missing_critical(df)
print(f'Dropped {before - len(df):,} rows missing critical fields ({(before - len(df)) / before * 100:.2f}%)')

Dropped 47,858 rows missing critical fields (9.57%)


## Step 4 — Impute missing weather values with column medians

Weather variables are right-skewed, so medians are more robust than means.

In [5]:
missing_before = df[list(NUMERIC_WEATHER_COLUMNS)].isna().sum()
df = impute_weather(df)
missing_after = df[list(NUMERIC_WEATHER_COLUMNS)].isna().sum()
pd.DataFrame({'missing_before': missing_before, 'missing_after': missing_after})

,missing_before,missing_after
temperature_f,9439,0
wind_chill_f,127714,0
humidity,10051,0
pressure_in,8083,0
visibility_mi,10179,0
wind_speed_mph,35345,0
precipitation_in,140342,0


## Step 5 — Fill categorical nulls with `Unknown`

Preserves rows for geographic and temporal aggregations.

In [6]:
df = impute_categorical(df)
cat_nulls = df[[c for c in CATEGORICAL_FILL_COLUMNS if c in df.columns]].isna().sum()
print('Remaining categorical nulls:')
print(cat_nulls)

Remaining categorical nulls:
wind_direction       0
weather_condition    0
zipcode              0
city                 0
county               0
street               0
timezone             0
sunrise_sunset       0
civil_twilight       0
dtype: int64


## Step 6 — Coerce infrastructure POI flags to boolean

In [7]:
df = coerce_booleans(df)
print('Boolean POI dtypes:')
print(df[[c for c in BOOLEAN_POI_COLUMNS if c in df.columns]].dtypes)

Boolean POI dtypes:
amenity            bool
bump               bool
crossing           bool
give_way           bool
junction           bool
no_exit            bool
railway            bool
roundabout         bool
station            bool
stop               bool
traffic_calming    bool
traffic_signal     bool
dtype: object


## Step 7 — Compute `duration_min` and drop impossible durations

Negative durations (end before start) or > 24 h are data-entry errors and are removed.

In [8]:
before = len(df)
df = add_duration(df)
print(f'Dropped {before - len(df):,} rows with invalid duration')
print('Duration distribution (min):')
print(df['duration_min'].describe().round(2))

Dropped 1,925 rows with invalid duration
Duration distribution (min):
count    450217.00
mean        106.07
std         124.02
min           3.00
25%          30.00
50%          62.18
75%         122.08
max        1440.00
Name: duration_min, dtype: float64


## Step 8 — Derive time-based features

`hour`, `day_of_week`, `day_name`, `month`, `month_name`, `year`, `season`, `is_weekend`, `is_rush_hour` — these feed the EDA charts and the Tableau dashboard filters.

In [9]:
df = add_time_features(df)
df[['start_time', 'hour', 'day_name', 'month_name', 'season', 'is_weekend', 'is_rush_hour']].head()

,start_time,hour,day_name,month_name,season,is_weekend,is_rush_hour
0,2021-11-24 19:36:02,19,Wednesday,November,Fall,False,True
1,2017-08-08 08:52:45,8,Tuesday,August,Summer,False,True
2,2016-10-24 16:09:01,16,Monday,October,Fall,False,True
3,2022-12-23 14:05:02,14,Friday,December,Winter,False,False
4,2017-10-01 16:44:33,16,Sunday,October,Fall,True,True


## Step 9 — Map severity to human-readable labels

| Severity | Label |
|---|---|
| 1 | Low |
| 2 | Moderate |
| 3 | High |
| 4 | Critical |

Also add `is_high_severity` (severity ≥ 3) for hypothesis testing.

In [10]:
df = add_severity_label(df)
df['severity_label'].value_counts()

severity_label
Moderate    349967
High         84320
Critical     11623
Low           4307
Name: count, dtype: int64

## Step 10 — Handle outliers

- Cap `distance_mi` at the 99th percentile (long-tail artefacts).
- Remove temperatures outside the physically possible range (-60 °F to 140 °F).

In [11]:
before = len(df)
df = handle_outliers(df)
print(f'Dropped {before - len(df):,} rows with impossible temperature values')
print('Distance distribution after capping:')
print(df['distance_mi'].describe().round(3))

Dropped 3 rows with impossible temperature values
Distance distribution after capping:
count    450214.000
mean          0.459
std           1.075
min           0.000
25%           0.000
50%           0.010
75%           0.385
max           6.682
Name: distance_mi, dtype: float64


## Step 11 — Drop exact duplicates and finalise

In [12]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f'Dropped {before - len(df):,} exact duplicates')
print(f'Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

Dropped 523 exact duplicates
Final shape: 449,691 rows × 48 columns


## Cleaning audit

Compare row and null counts before and after cleaning.

In [13]:
rows_final = len(df)
loss_pct = (rows_initial - rows_final) / rows_initial * 100
print(f'Initial rows : {rows_initial:,}')
print(f'Final rows   : {rows_final:,}')
print(f'Row loss     : {rows_initial - rows_final:,} ({loss_pct:.2f}%)')

remaining_nulls = df.isna().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]
print('\nRemaining nulls (should be empty or only in low-priority columns):')
print(remaining_nulls if not remaining_nulls.empty else '  none')

Initial rows : 500,000
Final rows   : 449,691
Row loss     : 50,309 (10.06%)

Remaining nulls (should be empty or only in low-priority columns):
  none


## Export cleaned dataset

In [14]:
save_processed(df, PROCESSED_PATH)
print(f'✓ Saved cleaned dataset → {PROCESSED_PATH}')
print(f'  Size on disk: {PROCESSED_PATH.stat().st_size / 1e6:.1f} MB')

✓ Saved cleaned dataset → /Users/aryankinha/Downloads/B_G14_DVACapstone-main/data/processed/cleaned_dataset.csv
  Size on disk: 150.2 MB
